In [1]:
pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 1.7 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


# ============================================================
# 1. 加载 MNIST 数据（与你之前的实验保持一致）
# ============================================================

print("Loading MNIST...")

mnist = fetch_openml('mnist_784', version=1)
X = mnist.data.to_numpy()
y = mnist.target.astype(int).to_numpy()

# 取 1000 训练，300 测试（老师要求：小数据训练）
X_small, _, y_small, _ = train_test_split(
    X, y, train_size=1000, stratify=y, random_state=0
)
X_test_small, _, y_test_small, _ = train_test_split(
    X, y, train_size=300, stratify=y, random_state=1
)

print("Train subset:", X_small.shape)
print("Test subset:", X_test_small.shape)


# ============================================================
# 2. 图像 reshape → pad → 转为 torch tensors
# ============================================================

# MNIST: (N, 784) → (N, 1, 28, 28)
X_small = X_small.reshape((-1, 1, 28, 28))
X_test_small = X_test_small.reshape((-1, 1, 28, 28))

# pad 到 32×32（LeNet1 要求）
X_small = np.pad(
    X_small, ([0,0], [0,0], [2,2], [2,2]), mode='constant'
)
X_test_small = np.pad(
    X_test_small, ([0,0], [0,0], [2,2], [2,2]), mode='constant'
)

# 转成 Pytorch float32 / int64（很重要）
X_training = Variable(torch.from_numpy(X_small.astype('float32')))
y_training = Variable(torch.from_numpy(y_small.astype('int64')))

X_testing = Variable(torch.from_numpy(X_test_small.astype('float32')))
y_testing = Variable(torch.from_numpy(y_test_small.astype('int64')))

print("Final training shape:", X_training.shape)
print("Final testing shape:", X_testing.shape)


# ============================================================
# 3. 定义 LeNet1（老师给的架构）
# ============================================================

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 3)
        self.conv2 = nn.Conv2d(6, 16, 3)
        self.fc1 = nn.Linear(16 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features


# ============================================================
# 4. 定义损失函数 & 优化器（稳定学习率）
# ============================================================

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)   # 学习率调小，防止爆炸

batch_size = 32
EPOCHS = 10

print("\nStart training...\n")


# ============================================================
# 5. 训练循环（批训练，稳定可靠）
# ============================================================

for epoch in range(EPOCHS):
    running_loss = 0.0
    
    # 打乱顺序
    permutation = torch.randperm(X_training.size()[0])
    
    for i in range(0, X_training.size(0), batch_size):
        idx = permutation[i:i+batch_size]
        inputs = X_training[idx]
        labels = y_training[idx]

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}   Loss = {running_loss:.4f}")

print("\nTraining finished!\n")


# ============================================================
# 6. 测试集准确率
# ============================================================

correct = 0
total = 0

with torch.no_grad():
    for i in range(len(X_testing)):
        outputs = net(X_testing[i:i+1])
        _, predicted = torch.max(outputs.data, 1)
        total += 1
        correct += (predicted == y_testing[i]).item()

accuracy = correct / total
print(f"Test Accuracy = {accuracy:.4f}")

Loading MNIST...
Train subset: (1000, 784)
Test subset: (300, 784)
Final training shape: torch.Size([1000, 1, 32, 32])
Final testing shape: torch.Size([300, 1, 32, 32])

Start training...

Epoch 1/10   Loss = 53.3824
Epoch 2/10   Loss = 18.0054
Epoch 3/10   Loss = 9.6124
Epoch 4/10   Loss = 5.9598
Epoch 5/10   Loss = 3.8227
Epoch 6/10   Loss = 4.2501
Epoch 7/10   Loss = 1.3829
Epoch 8/10   Loss = 0.8802
Epoch 9/10   Loss = 0.4177
Epoch 10/10   Loss = 0.2138

Training finished!

Test Accuracy = 0.9533
